In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer



In [2]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    return embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
def build_doc_text(row):
    return f"""
[PROMPT]
{row['prompt']}

[ESSAY]
{row['essay']}

[EVALUATION]
{row['evaluation']}

[QUALITY SUMMARY]
TR={row['TR']}, CC={row['CC']}, LR={row['LR']}, GRA={row['GRA']}, overall={row['band']},
essay_len={row['essay_len']}, prompt_relevance={row['prompt_relevance']},
lexical_diversity={row['lexical_diversity']}, readability_score={row['readability_score']}
""".strip()

df = pd.read_csv("C:\\Users\\Admin\\IELTS-Writing-Evals\\ielts_train_df.csv")
df["doc_test"] = df.apply(build_doc_text, axis = 1)

In [5]:
def build_query_text(prompt, essay, pred_scores, feat_dict):
    return f"""
Find essays with similar writing quality.

[PROMPT]
{prompt}

[ESSAY]
{essay}

[PREDICTED SCORES]
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}

[FEATURES]
essay_len={feat_dict['essay_len']},
prompt_relevance={feat_dict['prompt_relevance']},
lexical_diversity={feat_dict['lexical_diversity']},
readability_score={feat_dict['readability_score']}
""".strip()

In [4]:
def normalize_diff(a, b, scale):
    return abs(a - b) / scale

def quality_distance(row, pred_scores, feat_dict):
    score_dist = (
        normalize_diff(row["TR"], pred_scores["TR"], 9.0) +
        normalize_diff(row["CC"], pred_scores["CC"], 9.0) +
        normalize_diff(row["LR"], pred_scores["LR"], 9.0) +
        normalize_diff(row["GRA"], pred_scores["GRA"], 9.0)
    )

    feat_dist = (
        normalize_diff(row["essay_len"], feat_dict["essay_len"], 400.0) +
        normalize_diff(row["prompt_relevance"], feat_dict["prompt_relevance"], 1.0) +
        normalize_diff(row["lexical_diversity"], feat_dict["lexical_diversity"], 1.0) +
        normalize_diff(row["readability_score"], feat_dict["readability_score"], 100.0)
    )

    return score_dist + 0.5 * feat_dist

In [6]:
def retrieve_cases(
    query_embedding,
    db_embeddings,
    df,
    pred_scores,
    feat_dict,
    top_k_vector=20,
    top_k_final=5,
):
    sims = cosine_similarity(query_embedding.reshape(1, -1), db_embeddings)[0]

    candidate_idx = np.argsort(-sims)[:top_k_vector]
    candidates = df.iloc[candidate_idx].copy()
    candidates["vector_sim"] = sims[candidate_idx]

    candidates["quality_dist"] = candidates.apply(
        lambda row: quality_distance(row, pred_scores, feat_dict),
        axis=1
    )

    candidates["final_score"] = candidates["vector_sim"] - 0.7 * candidates["quality_dist"]
    candidates = candidates.sort_values("final_score", ascending=False)

    return candidates.head(top_k_final)